<a href="https://colab.research.google.com/github/kyuuf/youtubeclass/blob/master/LDA_%EB%AA%A8%EB%8D%B8%EB%A7%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

YouTube Korean Comment LDA Topic Modeling (Colab)

In [ ]:
!pip install konlpy gensim pyLDAvis pandas

# === 1️⃣ 필요한 라이브러리 불러오기 ===
from konlpy.tag import Okt
from gensim import corpora, models
import pandas as pd
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
import re

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.0/438.0 kB 19.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


필수 라이브러리 설치 및 폰트 설정 (이 셀을 가장 먼저 실행해 주세요!)

In [ ]:

!pip install konlpy > /dev/null  # 형태소 분석기
!pip install pyLDAvis > /dev/null # 시각화 도구
!apt-get update -qq
!apt-get install fonts-nanum* -qq # 한글 폰트 설치

# 폰트 매니저 재설정 (설치 후 폰트 적용을 위해)
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import matplotlib as mpl

# 나눔고딕 폰트 경로 설정
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rc('font', family=font_name)
mpl.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지

print("라이브러리 및 한글 폰트 설치 완료!")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Selecting previously unselected package fonts-nanum.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Selecting previously unselected package fonts-nanum-extra.
Preparing to unpack .../fonts-nanum-extra_20200506-1_all.deb ...
Unpacking fonts-nanum-extra (20200506-1) ...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Setting up fonts-nanum-extra (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Setting up fonts-nanum-coding (2.5-3) ...
Setting up fonts-nanum-eco (1.000-7) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...
라이브러리 및 한글 폰트 설치 완료!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
import re
import pandas as pd
from konlpy.tag import Okt
from gensim import corpora, models
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
from google.colab import files # Colab 파일 다운로드 기능

# --- 파일 이름 설정 (업로드한 파일명과 똑같이 적어주세요!) ---
FILE_PATH = "youtube_all_comments (1).csv"

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

LDA 분석

In [ ]:
try:
    # 파일 불러오기
    df = pd.read_csv(FILE_PATH, encoding='CP949')
except UnicodeDecodeError:
    df = pd.read_csv(FILE_PATH, encoding='utf-8') # 인코딩 에러 시 utf-8로 재시도
except FileNotFoundError:
    print("❌ 오류: 파일을 찾을 수 없습니다. 왼쪽 폴더 아이콘을 눌러 파일을 업로드했는지 확인해주세요.")
    raise

# 한 줄 댓글로 정제
df['text'] = df['text'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
comments = df['text'].dropna().tolist()
print(f"✅ 총 {len(comments)}개의 댓글 로드 완료")

# 형태소 분석기 초기화
okt = Okt()

# 불용어 리스트
stop_words = [
    '이', '그', '저', '것', '수', '등', '들', '좀', '잘', '걍', '죠', '임',
    '거', '때', '때문', '너무', '정말', '진짜', '에서', '에게', '으로', '하다',
    'ㅅㅂ', '이다', '있다', '없다', '같다', '그리고', '하지만', '그냥', '또',
    '존나', '왜', '뭐', '이거', '저거', '영상', '댓글', '유튜브', '사람','손흥민','쏘니','소니','흥미니'
]

# 전처리 함수
def preprocess_korean(text):
    text = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', text) # 한글만 남기기
    tokens = okt.nouns(text) # 명사 추출
    tokens = [t for t in tokens if len(t) > 1 and t not in stop_words]
    return tokens

# ★ 데이터 정렬 (원문과 토큰화 데이터 짝 맞추기) ★
print("⏳ 형태소 분석 중입니다... (데이터 양에 따라 시간이 걸릴 수 있습니다)")
tokenized_data = [preprocess_korean(comment) for comment in comments]

# 빈 결과 제거 및 원문 필터링
valid_indices = [i for i, tokens in enumerate(tokenized_data) if len(tokens) > 0]
processed_comments = [tokenized_data[i] for i in valid_indices]
valid_comments = [comments[i] for i in valid_indices] # 원문도 똑같이 남김

print(f"✅ 전처리 완료! 분석 대상 댓글 수: {len(processed_comments)}")

# 사전 및 코퍼스 생성
dictionary = corpora.Dictionary(processed_comments)
corpus = [dictionary.doc2bow(text) for text in processed_comments]

# LDA 모델 학습
NUM_TOPICS = 3
lda_model = models.LdaModel(
    corpus,
    num_topics=NUM_TOPICS,
    id2word=dictionary,
    passes=10,
    random_state=42
)

# === 🌟 핵심 기능: 토픽별 대표 댓글 출력 함수 ===
def print_representative_comments(lda_model, corpus, texts, n=3):
    topic_info = []
    for i, row in enumerate(lda_model[corpus]):
        row = sorted(row, key=lambda x: (x[1]), reverse=True)
        if len(row) > 0:
            topic_num, prop_topic = row[0]
            topic_info.append([int(topic_num), prop_topic, texts[i]])

    topic_df = pd.DataFrame(topic_info, columns=['Topic_Num', 'Probability', 'Comment'])

    print("\n" + "="*50)
    print("📝 토픽별 대표 댓글 (확률이 높은 가장 전형적인 반응)")
    print("="*50)

    for i in range(lda_model.num_topics):
        # 토픽 키워드 가져오기
        keywords = ", ".join([word for word, prop in lda_model.show_topic(i, topn=5)])
        print(f"\n🧩 [Topic {i+1}] 주요 키워드: {keywords}")

        # 대표 댓글 추출
        top_comments = topic_df[topic_df['Topic_Num'] == i].sort_values('Probability', ascending=False).head(n)

        for idx, row in top_comments.iterrows():
            print(f"  👉 ({row['Probability']:.1%}) {row['Comment']}")

# 대표 댓글 출력 실행
print_representative_comments(lda_model, corpus, valid_comments, n=3)

# 시각화 저장 및 다운로드
vis = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.save_html(vis, 'korean_lda_topics.html')
print("\n✅ 시각화 파일(html) 생성이 완료되었습니다. 자동으로 다운로드됩니다.")
files.download('korean_lda_topics.html')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ 총 285개의 댓글 로드 완료


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

⏳ 형태소 분석 중입니다... (데이터 양에 따라 시간이 걸릴 수 있습니다)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
from gensim.models import CoherenceModel

# Coherence Score 계산
coherence_model_lda = CoherenceModel(
    model=lda_model,
    texts=processed_comments,  # 위 코드에서 전처리 완료된 데이터 변수명
    dictionary=dictionary,     # 위 코드에서 만든 사전
    coherence='c_v'
)

coherence_lda = coherence_model_lda.get_coherence()
print(f"현재 모델(토픽 4개)의 Coherence Score: {coherence_lda:.4f}")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

현재 모델(토픽 4개)의 Coherence Score: 0.3995


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

몇 개의 토픽이 최선일까?" 비교하기

In [ ]:
import matplotlib.pyplot as plt

def compute_coherence_values(dictionary, corpus, texts, limit, start=2, step=1):
    coherence_values = []
    model_list = []

    print("경우의 수 별로 학습을 시작합니다 (시간이 소요됩니다)...")

    for num_topics in range(start, limit, step):
        # 모델 생성
        model = models.LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=num_topics,
            passes=10,             # 학습 횟수
            random_state=42
        )
        model_list.append(model)

        # Coherence Score 계산
        coherencemodel = CoherenceModel(
            model=model,
            texts=texts,
            dictionary=dictionary,
            coherence='c_v'
        )
        coherence_values.append(coherencemodel.get_coherence())
        print(f"▶ 토픽 {num_topics}개 완료: 점수 {coherence_values[-1]:.4f}")

    return model_list, coherence_values

# 2개부터 3개까지 테스트
limit = 5; start = 2; step = 1
model_list, coherence_values = compute_coherence_values(
    dictionary=dictionary,
    corpus=corpus,
    texts=processed_comments,
    start=start,
    limit=limit,
    step=step
)


# 가장 높은 점수의 토픽 개수 출력
max_score_idx = coherence_values.index(max(coherence_values))
optimal_num = start + max_score_idx
print(f"\n✅ 가장 높은 점수({max(coherence_values):.4f})를 기록한 토픽 개수는 '{optimal_num}개' 입니다.")

-=-------------------------------------------------------

In [ ]:
import re
import pandas as pd
from konlpy.tag import Okt
from gensim import corpora, models
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# === 2️⃣ 형태소 분석기(Okt) 초기화 ===
okt = Okt()

# === 3️⃣ 한글 CSV 파일 불러오기 ===
FILE_PATH = "영어(번역)종합본.csv"   # Colab에 업로드한 파일명과 동일해야 함
df = pd.read_csv(FILE_PATH, encoding='CP949')

# 🔧 (추가) 줄바꿈/여러 칸 공백을 한 칸으로 통일해서 "한 줄 댓글"로 만들기
df['text'] = (
    df['text']
      .astype(str)
      .str.replace(r'\s+', ' ', regex=True)  # \n, \t, 여러 공백 → 공백 1개
      .str.strip()
)

comments = df['text'].dropna().tolist()
print(f"✅ 총 {len(comments)}개의 댓글 로드 완료")

# === 4️⃣ 불용어 리스트 정의 ===
stop_words = [
    '이', '그', '저', '것', '수', '등', '들', '좀', '잘', '걍', '죠', '임',
    '거', '때', '등', '너무', '정말', '진짜', '에서', '에게', '으로', '하다',
    '되다', '이다', '있다', '없다', '같다', '그리고', '하지만', '그러나', '또',
    '또한', '때문', '왜', '뭐', '이거', '저거', '영상', '댓글', '유튜브', '사람'
]

# === 5️⃣ 전처리 함수 정의 ===
def preprocess_korean(text):
    # ① 한글 이외 문자 제거 (영어, 숫자, 이모지, 특수기호 등 → 전부 삭제)
    text = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', text)

    # ② 형태소 분석 - 명사만 추출
    tokens = okt.nouns(text)

    # ③ 불용어 제거 + 한 글자 제거
    tokens = [t for t in tokens if len(t) > 1 and t not in stop_words]
    return tokens

# === 6️⃣ 전체 댓글 전처리 ===
processed_comments = [preprocess_korean(comment) for comment in comments]

# 🔧 (추가) 완전히 빈 문서는 제거 (LDA에서 쓸모 없음)
processed_comments = [tokens for tokens in processed_comments if len(tokens) > 0]

print(f"✅ 토큰이 있는 댓글 수: {len(processed_comments)}")

# 혹시 전처리 후 문서가 하나도 없으면 종료
if len(processed_comments) == 0:
    raise ValueError("전처리 후 토큰이 남은 댓글이 없습니다. 불용어/전처리 규칙을 완화해 주세요.")

# === 7️⃣ LDA를 위한 사전(Dictionary)과 코퍼스(Corpus) 생성 ===
dictionary = corpora.Dictionary(processed_comments)
corpus = [dictionary.doc2bow(text) for text in processed_comments]

# === 8️⃣ LDA 모델 학습 ===
NUM_TOPICS = 5  # 주제 개수 조절 가능
lda_model = models.LdaModel(
    corpus,
    num_topics=NUM_TOPICS,
    id2word=dictionary,
    passes=10,
    random_state=42
)

# === 9️⃣ 주요 토픽 출력 ===
print("\n🎯 주요 토픽 결과:")
for idx, topic in lda_model.show_topics(formatted=False):
    words = ", ".join([word for word, _ in topic])
    print(f"\n🧩 Topic {idx+1}: {words}")

# === 🔟 시각화 결과 저장 ===
vis = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.save_html(vis, 'korean_lda_topics.html')
print("\n✅ 시각화 결과 'korean_lda_topics.html' 파일로 저장 완료!")

# === 11️⃣ Colab에서 결과 파일 다운로드 ===
from google.colab import files
files.download('korean_lda_topics.html')


In [ ]:
import re
import pandas as pd
from konlpy.tag import Okt
from gensim import corpora, models
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# === 2️⃣ 형태소 분석기(Okt) 초기화 ===
okt = Okt()

# === 3️⃣ 한글 CSV 파일 불러오기 ===
FILE_PATH = "영어(번역)종합본.csv"   # Colab에 업로드한 파일명과 동일해야 함
df = pd.read_csv(FILE_PATH, encoding='CP949')

# 🔧 (추가) 줄바꿈/여러 칸 공백을 한 칸으로 통일해서 "한 줄 댓글"로 만들기
df[1] = (
    df[1]
      .astype(str)
      .str.replace(r'\s+', ' ', regex=True)  # \n, \t, 여러 공백 → 공백 1개
      .str.strip()
)

comments = df[1].dropna().tolist()
print(f"✅ 총 {len(comments)}개의 댓글 로드 완료")

# === 4️⃣ 불용어 리스트 정의 ===
stop_words = [
    '이', '그', '저', '것', '수', '등', '들', '좀', '잘', '걍', '죠', '임',
    '거', '때', '등', '너무', '정말', '진짜', '에서', '에게', '으로', '하다',
    '되다', '이다', '있다', '없다', '같다', '그리고', '하지만', '그러나', '또',
    '또한', '때문', '왜', '뭐', '이거', '저거', '영상', '댓글', '유튜브', '사람'
]

# === 5️⃣ 전처리 함수 정의 ===
def preprocess_korean(text):
    # ① 한글 이외 문자 제거 (영어, 숫자, 이모지, 특수기호 등 → 전부 삭제)
    text = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', text)

    # ② 형태소 분석 - 명사만 추출
    tokens = okt.nouns(text)

    # ③ 불용어 제거 + 한 글자 제거
    tokens = [t for t in tokens if len(t) > 1 and t not in stop_words]
    return tokens

# === 6️⃣ 전체 댓글 전처리 ===
processed_comments = [preprocess_korean(comment) for comment in comments]

# 🔧 (추가) 완전히 빈 문서는 제거 (LDA에서 쓸모 없음)
processed_comments = [tokens for tokens in processed_comments if len(tokens) > 0]

print(f"✅ 토큰이 있는 댓글 수: {len(processed_comments)}")

# 혹시 전처리 후 문서가 하나도 없으면 종료
if len(processed_comments) == 0:
    raise ValueError("전처리 후 토큰이 남은 댓글이 없습니다. 불용어/전처리 규칙을 완화해 주세요.")

# === 7️⃣ LDA를 위한 사전(Dictionary)과 코퍼스(Corpus) 생성 ===
dictionary = corpora.Dictionary(processed_comments)
corpus = [dictionary.doc2bow(text) for text in processed_comments]

# === 8️⃣ LDA 모델 학습 ===
NUM_TOPICS = 5  # 주제 개수 조절 가능
lda_model = models.LdaModel(
    corpus,
    num_topics=NUM_TOPICS,
    id2word=dictionary,
    passes=10,
    random_state=42
)

# === 9️⃣ 주요 토픽 출력 ===
print("\n🎯 주요 토픽 결과:")
for idx, topic in lda_model.show_topics(formatted=False):
    words = ", ".join([word for word, _ in topic])
    print(f"\n🧩 Topic {idx+1}: {words}")

# === 🔟 시각화 결과 저장 ===
vis = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.save_html(vis, 'korean_lda_topics.html')
print("\n✅ 시각화 결과 'korean_lda_topics.html' 파일로 저장 완료!")

# === 11️⃣ Colab에서 결과 파일 다운로드 ===
from google.colab import files
files.download('korean_lda_topics.html')
